# Analysis, Plots and *Insights*

This *notebook* holds the charts and the *insights* drawn from the
*Dataframe* cleaned in the previous one.

<table align="left">
  <tr>
    <td>
      <a href="https://colab.research.google.com/github/ValentimPiazera/EDA-Carros-Eletricos-Washington/blob/main/notebooks/II-analysis.ipynb" target="_parent">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
      </a>
    </td>
    <td>
      <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ValentimPiazera/EDA-Carros-Eletricos-Washington/blob/main/notebooks/II-analysis.ipynb">
        <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/>
      </a>
    </td>
  </tr>
</table>

> **Note:** this notebook is written to run from the **repository root**, not from `notebooks/`.
> Locally, `.vscode/settings.json` already points VS Code's Jupyter root at the workspace folder;
> from a terminal, launch Jupyter from the repository root.
>
> If you're running on Google Colab or Kaggle, run the cell below first: it clones the repository,
> which is what brings in `data/processed/` — the file exported by `I-cleaning`.
>
> On **Kaggle**, make sure Internet access is enabled: *Settings → Internet → On*.

In [1]:
# Setup: clone repo when running on Colab/Kaggle (skip if running locally!)
import os

REPO = "EDA-Carros-Eletricos-Washington"

IN_COLAB_OR_KAGGLE = (
    "google.colab" in str(get_ipython())
    or "kaggle" in os.environ.get("KAGGLE_URL_BASE", "").lower()
)

# The second test is what makes the cell re-runnable: once the working
# directory is the repository root there is nothing left to clone or enter.
if IN_COLAB_OR_KAGGLE and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        !git clone https://github.com/ValentimPiazera/{REPO}.git
    # The repo root becomes the working directory, so the `data/` paths
    # and `from src import cleaning` resolve as they do locally.
    %cd {REPO}


## Part I — Loading and Imports

Much like the section of the same name in the previous *notebook*, except
that this time I am only checking that everything done there worked, since I
already know the data.

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src import utils, viz

In [3]:
df = pd.read_csv("data/processed/ev_population_washington_clean.csv")
df.head(10)

,County,City,Model Year,Make,Model,Electric Vehicle Type,CAFV Status,Electric Range,Electric Utility,Vehicle Age,Longitude,Latitude
0,Kitsap,Bainbridge Island,2018,TESLA,MODEL 3,BEV,Eligible,215.0,Puget Sound Energy,8,-122.52100,47.62732
1,Kitsap,Port Orchard,2011,NISSAN,LEAF,BEV,Eligible,73.0,Puget Sound Energy,15,-122.70348,47.52028
2,King,Bothell,2011,NISSAN,LEAF,BEV,Eligible,73.0,Puget Sound Energy,15,-122.20563,47.76144
3,Yakima,Selah,2020,CHEVROLET,BOLT EV,BEV,Eligible,259.0,Pacificorp,6,-120.53145,46.65405
4,King,Seattle,2020,KIA,NIRO,PHEV,Not Eligible (Low Range),26.0,City Of Seattle,6,-122.33364,47.73709
5,Thurston,Olympia,2023,KIA,NIRO,PHEV,Eligible,33.0,Puget Sound Energy,3,-122.92333,47.03779
6,King,Seattle,2016,AUDI,A3,PHEV,Not Eligible (Low Range),16.0,City Of Seattle,10,-122.35436,47.67596
7,Thurston,Tumwater,2018,TESLA,MODEL 3,BEV,Eligible,215.0,Puget Sound Energy,8,-123.04078,46.94792
8,Kitsap,Bremerton,2018,TESLA,MODEL S,BEV,Eligible,249.0,Puget Sound Energy,8,-122.62749,47.56500
9,Snohomish,Lynnwood,2017,CHEVROLET,VOLT,PHEV,Eligible,53.0,Puget Sound Energy,9,-122.27981,47.85727


## Part II — Fleet Overview

This part holds charts with general information about the fleet, based on all
270 thousand rows of the *dataframe*, among them:

* **1 —** Which makes have the most vehicles registered, and whether that is
still true of the newest model years?

* **2 —** Which individual models, once the manufacturer names stop hiding
them?

* **3 —** BEV versus PHEV: how does the split move from one model year to the
next?

* **4 —** How is the fleet distributed by age?

* **5 —** And where in the state does it actually sit?

*CAFV* eligibility is deliberately not charted. The rule `Electric Range >= 30`
explains 100% of the statuses the DOL has ruled on, so the column says nothing
the range column does not already say — and a chart of it would show 37% of the
fleet without admitting it, because the eligibility of the other 63% was never
researched.

In [4]:
top_10_makes = utils.top_makes(df)

top_10_makes

,Make,Registrations
0,TESLA,110210
1,CHEVROLET,19015
2,NISSAN,15938
3,FORD,14908
4,KIA,13600
5,TOYOTA,11350
6,BMW,11176
7,HYUNDAI,9806
8,RIVIAN,8491
9,VOLKSWAGEN,7356


In [5]:
make_shares = utils.make_share_by_era(df)

fig = viz.top_makes_bar(make_shares)

fig.show()

### Figure 1 — Analysis

The two bars are the point. Read the fleet-wide one and Tesla holds 40.7% with
Chevrolet and Nissan behind it; read the recent one and the board changes.

Chevrolet falls from 7.0% of the fleet to 3.8% of model years 2024 onwards, and
Nissan from 5.9% to 2.7% — less than half. Both built their position on a single
car sold in volume years ago: the Bolt and the Leaf. Moving the other way,
Hyundai goes 3.6% → 5.4%, Kia 5.0% → 6.9%, Toyota 4.2% → 5.1% and Rivian 3.1% →
4.5%. Tesla itself gives up ground, 40.7% → 35.8%.

**Conclusion:** a ranking of the fleet is a ranking of a decade of accumulated
sales, and reading it as *who is winning* gets two of the top three wrong. The
gap between the bars is the more useful number: it separates the makes still
building a position from the ones living off an earlier car.

In [6]:
top_10_models = utils.top_models(df)

top_10_models

,Model,Electric Vehicle Type,Registrations
0,TESLA MODEL Y,BEV,57163
1,TESLA MODEL 3,BEV,36941
2,NISSAN LEAF,BEV,13434
3,CHEVROLET BOLT EV,BEV,7679
4,TESLA MODEL S,BEV,7663
5,TESLA MODEL X,BEV,6624
6,FORD MUSTANG MACH-E,BEV,6219
7,VOLKSWAGEN ID.4,BEV,5959
8,HYUNDAI IONIQ 5,BEV,5593
9,JEEP WRANGLER,PHEV,4947


In [7]:
fig = viz.top_models_bar(top_10_models)

fig.show()

### Figure 2 — Analysis

Aggregating by manufacturer hid where the concentration actually sits. Two
cars — the Model Y and the Model 3 — are **34.8%** of every electric vehicle
registered in Washington, and the ten models above are 56.3% of the fleet out
of 184 on the road.

Nine of the ten are battery-electric. The exception is the Jeep Wrangler at
4,947 registrations, which makes the best-selling plug-in hybrid in the state a
body-on-frame off-roader rather than a commuter car.

**Conclusion:** the market-share question is usually asked about brands, and
answering it about products gives a sharper number. Tesla's 40.7% of the fleet
is not spread across a range; it is two cars.

In [8]:
type_mix = utils.type_mix_by_year(df)

type_mix

,Model Year,Electric Vehicle Type,Registrations,Share,Year Total
0,2011,BEV,532,90.3,589
1,2011,PHEV,57,9.7,589
2,2012,BEV,609,43.7,1395
3,2012,PHEV,786,56.3,1395
4,2013,BEV,2460,62.1,3961
5,2013,PHEV,1501,37.9,3961
6,2014,BEV,1555,48.4,3210
7,2014,PHEV,1655,51.6,3210
8,2015,BEV,3151,71.8,4389
9,2015,PHEV,1238,28.2,4389


In [9]:
fig = viz.type_mix_bars(type_mix)

fig.show()

### Figure 3 — Analysis

Across the whole fleet the split is 79.8% battery-electric to 20.2% plug-in,
and that single ratio is the least interesting thing this variable does.

By model year it swings. Plug-ins were **46.4%** of model year 2017 — very
nearly half — collapsed to 12.8% by 2023, and then came back: 21.4% of model
year 2024 and 21.7% of 2025. The 2023 trough and the recovery after it are the
two facts a fleet-wide pie chart cannot show.

The recovery lines up with something figure 6 shows independently. Plug-in
median range sat at 25 miles from 2017 to 2021 and then climbed — 30 miles in
2022, 32 in 2023, 35 in 2025 — so the years in which plug-ins won share back
are the years in which they stopped being short-range compromises.

The chart stops at model year 2025 on purpose. A share taken from three weeks
of 2026 is not a smaller measurement of that year, it is a biased one: the
partial year reads 93.9% BEV, which is the calendar rather than the market.

**Conclusion:** the plug-in hybrid is not a transitional technology on its way
out in this dataset. It lost half its share and won a good part of it back
while its range grew by half.

In [10]:
df["Vehicle Age"].mean().round()

np.float64(4.0)

In [11]:
by_model_year = utils.registrations_by_year(df)

fig = viz.model_year_area(by_model_year)

fig.show()

### Figure 4 — Analysis

Two readings are possible here. Either people are adopting EVs with a
definitive boom from 2020 on, or people change cars often, in which case we
cannot tell whether their previous vehicle was already electric. The truth is
a little of both: the state offered incentives and tax exemptions to electric
cars shortly before the definitive boom, purchasing power in the country is
high, and getting around an American city by anything other than a car is
difficult.

Two cautions the shape does not carry on its own. This is a *stock* — every
bar counts a car still registered in January 2026, so the early years are
eroded by everything scrapped, sold on or exported since. And the fall after
2023 cannot be read as a slowdown: model year 2026 is three weeks long, as the
annotation says, and even 2025 may still be filling up. With no registration
date in the file, a genuine decline and a partial year look identical.

In [12]:
top_15_cities = utils.top_cities(df)

top_15_cities.head(15)

,City,Registrations
0,Everett,4266
1,Lynnwood,4425
2,Bellingham,4507
3,Spokane,4542
4,Kent,4622
5,Tacoma,5900
6,Olympia,6366
7,Renton,7448
8,Sammamish,7539
9,Kirkland,7702


In [13]:
points = utils.registrations_by_location(df)

fig = viz.registration_map(points)

fig.show()

### Figure 5 — Analysis

Washington's electric fleet is not distributed across Washington. **92.1%** of
it sits west of longitude 121°W — the Cascades — and the five counties around
Puget Sound hold **77.2%** between them. Six of the state's 39 counties hold
fewer than 100 electric vehicles each.

Each bubble is one of the 825 points the DOL geocodes to, sized by the
registrations on it. They are postal-area centroids rather than addresses, so
the map is precise about *where the clusters are* and deliberately vague about
any single car. The largest single point is not Seattle but Redmond, with
6,518 registrations; the ten largest points together hold only 14.5% of the
fleet, which is the sense in which this is a corridor rather than a capital.

**Conclusion:** the distribution follows population and income, not policy
boundaries. The map cannot say that Redmond adopts electric cars faster than
Spokane — for that it would need a per-capita denominator, which this file does
not carry. What it does show is that any statewide claim drawn from this
dataset is, in practice, a claim about the Puget Sound corridor.

## Part III — Range and Technological Evolution

I set this part aside for a general view of how the technology evolved. It
opens with the plainest question the data can be asked — how far does a car of
a given model year go — and then splits into two *scatter plots*, one per
period, setting volume against range so that a cluster of dots reads as an
average, a lot of range on few units reads as a luxury car, and vice versa.

The two windows are the ones figure 4 suggests: 2015 – 2019, the first boom,
and 2020 to 2026, the definitive one. Only models whose range the DOL has
actually researched reach those two charts, so that both axes of a point
describe the same vehicles — which, for the second window, turns out to be the
whole story.

In [14]:
range_per_year = utils.range_by_model_year(df)

range_per_year

,Model Year,Electric Vehicle Type,Registrations,Measured,Median Electric Range,Coverage
0,1999,BEV,2,2,74.0,1.000000
1,2000,BEV,8,8,58.0,1.000000
2,2002,BEV,1,1,95.0,1.000000
3,2003,BEV,1,1,95.0,1.000000
4,2008,BEV,20,19,220.0,0.950000
5,2010,BEV,21,21,245.0,1.000000
6,2010,PHEV,2,2,100.0,1.000000
7,2011,BEV,532,532,73.0,1.000000
8,2011,PHEV,57,57,35.0,1.000000
9,2012,BEV,609,609,73.0,1.000000


In [15]:
fig = viz.range_by_year_lines(range_per_year)

fig.show()

### Figure 6 — Analysis

Two lines, and the interesting one is the one that stops.

The BEV median sits at 73 miles for model year 2011 and holds there — 73, 75,
84, 84, 93 — until 2017, when it jumps to 210 and then climbs to 291 by 2020.
A fourfold rise, and every point of it resting on a model year the DOL
researched in full. Then the line ends. Not because the technology stopped,
but because the research did: 4.2% of model year 2021 BEVs carry a range at
all, and from 2022 on it is zero. This file cannot tell you how far a 2023
Model Y goes, and no amount of averaging will make it.

The plug-in line has no such gap — coverage sits at or near 100% for every
model year in the export — but it needs reading with more care. The swings up
to 2016 (35, 35, 19, 38, 19, 19) are composition, not technology: whichever
model sold most that year sets the median, and a Prius Plug-in rated at 6
miles and a Volt rated at 38 were on sale at the same time. From 2017 it
settles at 25 and holds there for five years, then climbs: 30 in 2022, 32 in
2023 and 2024, 35 in 2025, 38 in 2026.

Both lines start at 2011 because the model years before it hold a handful of
cars each — two plug-ins in 2010, one battery-electric car in 2002 — and a
median of one vehicle is not a trend.

**Conclusion:** the honest answer to *how has battery range evolved* is two
different answers. For plug-in hybrids, a flat stretch and then a steady climb
to roughly double the 2015 figure, measured throughout and trustworthy. For
battery-electric cars, a fourfold rise up to 2020 and then silence — a gap in
the DOL's research rather than in the market. A chart that draws a BEV line
across 2021-2026 is inventing it, which is why this one does not.

In [16]:
models_2015_2019 = utils.models_by_period(df, 2015, 2019)

models_2015_2019

,Model,Mean Electric Range,Registrations,Coverage
0,AUDI A3 PHEV,16.0,532,1.000000
1,AUDI E-TRON BEV,204.0,529,1.000000
2,BMW 330E PHEV,14.0,179,1.000000
3,BMW 530E PHEV,15.0,288,1.000000
4,BMW 740E PHEV,14.0,26,1.000000
5,BMW I3 BEV,104.0,387,1.000000
6,BMW I3 PHEV,90.0,961,1.000000
7,BMW I8 PHEV,15.0,87,1.000000
8,BMW X5 PHEV,14.0,465,1.000000
9,CADILLAC CT6 PHEV,31.0,13,1.000000


In [17]:
fig = viz.early_period_scatter(models_2015_2019)

fig.show()

### Figure 7 — Analysis

We see a fairly diverse market, with countless PHEVs in the bottom left corner
and BEV models from Tesla, Chevrolet, Hyundai and Nissan near the top of the
range axis — but again Tesla dominates, placing many models at the top centre
and the top right. Small dots also show up in the top left corner, which are
BEVs from luxury makes such as Jaguar and Audi.

Worth holding on to for the comparison that follows: this window needs no
qualification at all. All 55 models and all 42,448 registrations between 2015
and 2019 carry a researched range, so every dot here is measured rather than
inferred.

In [18]:
models_2020_2026 = utils.models_by_period(df, 2020, 2026)

models_2020_2026

,Model,Mean Electric Range,Registrations,Coverage
1,ALFA ROMEO TONALE PHEV,33.0,98,1.0
3,AUDI A7 E PHEV,24.0,11,1.0
4,AUDI A8 E PHEV,17.0,4,1.0
9,AUDI Q5 PHEV,23.0,160,1.0
10,AUDI Q5 E PHEV,22.0,1312,1.0
...,...,...,...,...
153,VOLVO S60 PHEV,33.0,196,1.0
154,VOLVO S90 PHEV,35.0,17,1.0
155,VOLVO V60 PHEV,39.0,100,1.0
157,VOLVO XC60 PHEV,30.0,1658,1.0


In [19]:
fig = viz.late_period_scatter(models_2020_2026)

fig.show()

### Figure 8 — Analysis

This chart is missing most of the cars people actually bought, and that is the
finding rather than a defect in it.

Of the 218,796 vehicles carrying a 2020-2026 model year, only 37,813 — **17%**
— belong to a model the DOL researched well enough to place on this plane.
Every high-volume BEV is absent: the Model Y and its 57,163 registrations, the
Model 3, the Ioniq 5, the Mach-E. What remains is the plug-in half of the
market, where coverage is complete, clustered exactly where figure 6 says it
should be, between roughly 20 and 45 miles.

Set that against figure 7, which draws every model and every registration of
its window. The difference between the two charts is not the market changing
shape; it is the dataset going quiet.

**Conclusion:** for model years 2020 onwards this file supports statements
about plug-in hybrids and almost none about battery-electric cars. The reading
this figure used to carry — that Teslas are the kings of range — was never
measured here: it was a model year 2020 figure attached to a bubble sized by
six further years of registrations.

## Part IV — Market Share

Figure 1 shows Tesla ahead by a wide margin, and figures 7 and 8 show it
placing more models than anyone else. The table below turns that into
percentages across all 47 makes — and figure 9 then asks the question the
table cannot answer, which is whether that share is where the market is or
only where it has been.

In [20]:
share_by_make = utils.market_share(df)

share_by_make

,Make,Share
0,TESLA,40.7
1,CHEVROLET,7.0
2,NISSAN,5.9
3,FORD,5.5
4,KIA,5.0
5,TOYOTA,4.2
6,BMW,4.1
7,HYUNDAI,3.6
8,RIVIAN,3.1
9,VOLKSWAGEN,2.7


### What the Share Table Says

Tesla holds 40.7% of the fleet on its own. The top three makes hold 53.7%
between them and the top ten hold 82.0%, which leaves 18.0% to be shared out
among the remaining 37 makes.

The treemap that used to sit here is gone. Its top ten was the same ten makes
as figure 1, in the same order, and rounding each share to one decimal sent
twelve makes to 0.0% — an area of nothing, for brands that do exist.

**Conclusion:** the concentration is real, but it is a concentration of
*stock*. Every share above counts a car registered in January 2026 no matter
when it was built, so it describes a decade of accumulated sales rather than
the market as it stands today. Nothing in this notebook breaks that share down
by model year, which is exactly what would be needed to tell the two apart.

In [21]:
concentration = utils.concentration_by_year(df)

concentration

,Model Year,Leading make,Leader share,Top 3 share,Makes present
0,2011,NISSAN,87.4,98.5,5
1,2012,NISSAN,31.8,88.2,7
2,2013,NISSAN,41.6,76.1,7
3,2014,CHEVROLET,20.3,56.2,13
4,2015,NISSAN,36.6,69.1,12
5,2016,TESLA,29.1,62.5,16
6,2017,CHEVROLET,34.9,64.2,18
7,2018,TESLA,54.4,72.2,20
8,2019,TESLA,41.6,66.8,21
9,2020,TESLA,56.3,76.0,19


In [22]:
fig = viz.concentration_lines(concentration)

fig.show()

### Figure 9 — Analysis

This is the figure the share table cannot draw, and it reverses the reading.

The leading make's share of a single model year peaked at **56.3% in 2020** and
has fallen every year since: 54.4%, 47.1%, 44.4%, 32.3% and **26.4% for model
year 2025**. The top three together went from 76.0% to 40.0% over the same
stretch. Behind that, the grey bars: 19 makes present in model year 2020
against **37 in 2025**, with eight of them arriving in 2024 alone.

Concentration and variety moved in opposite directions, which is what a market
opening up looks like. The 40.7% in the table above is not wrong — it is simply
the average of a decade in which the first half was far more concentrated than
the second.

Model year 2026 is left out for the same reason as figure 3, and here the
distortion is worse: three weeks of registrations read 75.2% for the leader,
which would draw a spike undoing the entire trend.

**Conclusion:** Washington's electric market concentrated until 2020 and has
been dissolving that concentration ever since. Any claim of a monopoly drawn
from this dataset is a claim about the stock of cars, not about the market that
is producing them.

## Final Verdict

Washington's electric vehicle market is technologically mature and, read as a
stock of cars, heavily concentrated: Tesla holds 40.7% of the fleet and two of
its models alone are a third of every electric car in the state.

Read by model year, that concentration is a description of the past. The
leading make's share peaked at 56.3% in model year 2020 and has fallen every
year since, to 26.4% in 2025, while the number of makes on sale went from 19 to
37. The diversification that the accumulated figure would have set as a
challenge for the coming years is already well under way — it is simply
invisible to any chart that counts the whole fleet at once.

Two things this file cannot settle, and which any stronger claim would need.
It has no registration date, so every time axis here is the year a car was
built and every count is of the survivors still registered in January 2026.
And it stopped researching battery-electric range after model year 2020, so
the statement *range is no longer the constraint* — true of plug-ins, whose
median has grown by half since 2017 — rests on nothing at all for the
battery-electric cars that are 79.8% of the fleet.